# SteelVision — 00 · EDA on prepared detection datasets

Run locally after `make data` (or `python -m steelvision.data.prepare`).

Answers that drive modeling decisions:
- class balance per split (class imbalance handling in Phase 2),
- defect box size distribution — share of **small objects** (< 0.02% of image area,
  roughly < 32px at 640px input) → whether `imgsz=1024` or SAHI is needed,
- images / boxes per split.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_ROOT = Path("../data/processed")  # repo root relative to notebooks/
DATASETS = ["neu_det", "gc10"]

rows = []
for name in DATASETS:
    labels_dir = DATA_ROOT / name / "labels"
    if not labels_dir.exists():
        print(f"skip {name}: run prepare first")
        continue
    for split_dir in labels_dir.iterdir():
        for txt in split_dir.glob("*.txt"):
            for line in txt.read_text().splitlines():
                if not line.strip():
                    continue
                cls, xc, yc, w, h = line.split()
                rows.append(
                    {
                        "dataset": name,
                        "split": split_dir.name,
                        "image": txt.stem,
                        "class_id": int(cls),
                        "rel_area": float(w) * float(h),
                        "aspect": float(w) / float(h) if float(h) > 0 else 0.0,
                    }
                )
df = pd.DataFrame(rows)
df.head()

In [ ]:
import yaml

for name in DATASETS:
    data_yaml = DATA_ROOT / name / "data.yaml"
    if not data_yaml.exists():
        continue
    names = yaml.safe_load(data_yaml.read_text())["names"]
    sub = df[df.dataset == name]
    sub["class"] = sub.class_id.map(names)

    counts = sub.groupby(["split", "class"]).size().unstack(fill_value=0)
    print(f"=== {name}: boxes per split/class ===")
    display(counts)

    n_images = sub.groupby("split")["image"].nunique()
    print(f"images per split: {n_images.to_dict()}")

In [ ]:
import matplotlib.pyplot as plt

SMALL_AREA = 0.0002  # ~32x32 px at 640px input

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name in DATASETS:
    sub = df[df.dataset == name]
    if sub.empty:
        continue
    axes[0].hist(sub.rel_area, bins=60, alpha=0.5, label=name, range=(0, 0.2))
    axes[1].hist(sub.aspect, bins=60, alpha=0.5, label=name, range=(0, 10))
axes[0].axvline(SMALL_AREA, color="r", ls="--", label="small-object threshold")
axes[0].set_title("defect box relative area")
axes[0].legend()
axes[1].set_title("defect box aspect ratio (w/h)")
axes[1].legend()
plt.tight_layout()
plt.show()

for name in DATASETS:
    sub = df[df.dataset == name]
    if sub.empty:
        continue
    share = (sub.rel_area < SMALL_AREA).mean() * 100
    print(f"{name}: {share:.1f}% of boxes are small (rel. area < {SMALL_AREA})")

## Decisions this notebook feeds
- **Input size**: if the small-object share is high (> ~15%), train the next run at `imgsz=1024` before anything else.
- **Class imbalance**: classes with < half the median box count get oversampling or copy-paste augmentation (one change at a time, logged in W&B).
- **SAHI (optional extension)**: only worth it if small defects dominate test errors in the Phase 2 error analysis.